In [0]:
import requests, json, time
from datetime import datetime

api_key = "BU3A9YINQRU849P3"
symbols = ["AAPL", "MSFT", "GOOGL", "NVDA"]

raw_path = "/Volumes/jarvis_training/default/stock_api"

dbutils.fs.rm(f"{raw_path}/daily", True)
dbutils.fs.rm(f"{raw_path}/quote", True)
dbutils.fs.rm(f"{raw_path}/company", True)

dbutils.fs.mkdirs(f"{raw_path}/daily")
dbutils.fs.mkdirs(f"{raw_path}/quote")
dbutils.fs.mkdirs(f"{raw_path}/company")

for symbol in symbols:
    file_time = datetime.now().strftime("%Y%m%d_%H%M%S")

    daily_url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&outputsize=compact&apikey={api_key}"
    quote_url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey={api_key}"
    company_url = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={api_key}"

    daily_data = requests.get(daily_url).json()
    time.sleep(15)

    quote_data = requests.get(quote_url).json()
    time.sleep(15)

    company_data = requests.get(company_url).json()
    time.sleep(15)

    daily_records = []
    for date, v in daily_data.get("Time Series (Daily)", {}).items():
        daily_records.append({
            "symbol": symbol,
            "trade_date": date,
            "open": v.get("1. open"),
            "high": v.get("2. high"),
            "low": v.get("3. low"),
            "close": v.get("4. close"),
            "volume": v.get("5. volume")
        })

    quote = quote_data.get("Global Quote", {})
    clean_quote = {
        "symbol": quote.get("01. symbol"),
        "price": quote.get("05. price"),
        "volume": quote.get("06. volume"),
        "latest_trading_day": quote.get("07. latest trading day"),
        "previous_close": quote.get("08. previous close"),
        "price_change": quote.get("09. change"),
        "change_percent": quote.get("10. change percent")
    }

    clean_company = {
        "symbol": company_data.get("Symbol"),
        "company_name": company_data.get("Name"),
        "sector": company_data.get("Sector"),
        "industry": company_data.get("Industry"),
        "market_cap": company_data.get("MarketCapitalization"),
        "country": company_data.get("Country"),
        "exchange": company_data.get("Exchange")
    }

    dbutils.fs.put(
        f"{raw_path}/daily/{symbol}_{file_time}.json",
        "\n".join(json.dumps(r) for r in daily_records),
        True
    )

    dbutils.fs.put(
        f"{raw_path}/quote/{symbol}_{file_time}.json",
        json.dumps(clean_quote),
        True
    )

    dbutils.fs.put(
        f"{raw_path}/company/{symbol}_{file_time}.json",
        json.dumps(clean_company),
        True
    )

    print(f"{symbol} loaded")

Wrote 14899 bytes.
Wrote 182 bytes.
Wrote 178 bytes.
AAPL loaded
Wrote 14900 bytes.
Wrote 185 bytes.
Wrote 195 bytes.
MSFT loaded
Wrote 14999 bytes.
Wrote 186 bytes.
Wrote 212 bytes.
GOOGL loaded
Wrote 14999 bytes.
Wrote 185 bytes.
Wrote 181 bytes.
NVDA loaded
